In [10]:
# XGBoost Anomaly Detection on AnoShift Dataset

## 1. Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
import xgboost as xgb
import time
import os
import sys

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
test_dataset_year = [2006]
dataset_years = [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]
anoshift_db_path = '/AnoShift-RIT/datasets/Kyoto-2016_AnoShift/'  # Change this to your AnoShift database path

In [12]:
def load_data(anoshift_db_path, year):
    df = pd.read_parquet(os.path.join(anoshift_db_path, f'subset/{year}_subset.parquet'))
    
    # print(f"Loading {year} subset", df)
    return df

In [13]:
def rename_columns(df):
    categorical_cols = ["0", "1", "2", "3", "13"]
    numerical_cols = ["4", "5", "6", "7", "8", "9", "10", "11", "12"]
    timestamp_col = ["14"]
    additional_cols = ["15", "16", "17", "19"]
    label_col = ["18"]

    new_names = []
    for col_name in df.columns.values:
        if col_name in numerical_cols:
            df[col_name] = pd.to_numeric(df[col_name])
            new_names.append((col_name, "num_" + col_name))
        elif col_name in categorical_cols:
            new_names.append((col_name, "cat_" + col_name))
        elif col_name in timestamp_col:
            new_names.append((col_name, "timestamp_" + col_name))
        elif col_name in additional_cols:
            new_names.append((col_name, "bonus_" + col_name))
        elif col_name in label_col:
            df[col_name] = pd.to_numeric(df[col_name])
            new_names.append((col_name, "label"))
        else:
            new_names.append((col_name, col_name))
    df.rename(columns=dict(new_names), inplace=True)

    # df.loc[df['label'] < 0, 'label'] = -1
    # df['label'].replace({1:0}, inplace=True)
    # df['label'].replace({-1:1}, inplace=True)

    print("Renamed columns:", df)

    return df

In [14]:
def preprocess(df, enc=None):
    if not enc:
        enc = OneHotEncoder(handle_unknown='ignore')
        enc.fit(df.loc[:,['cat_' in i for i in df.columns]])
    
    num_cat_features = enc.transform(df.loc[:,['cat_' in i for i in df.columns]]).toarray()

    df_catnum = pd.DataFrame(num_cat_features)
    df_catnum = df_catnum.add_prefix('catnum_')

    df = df.reset_index(drop=True)
    df_new = pd.concat([df, df_catnum], axis=1)
   
    df_new.loc[df_new['label'] < 0, 'label'] = -1
    df_new['label'].replace({1:0}, inplace=True)
    df_new['label'].replace({-1:1}, inplace=True)

    print(df_new, enc)
    
    return df_new, enc

In [15]:
def extract_features_labels_timestamps(df):
    # Define columns
    feature_cols = [col for col in df.columns if col.startswith('num_') or col.startswith('catnum_')]
    timestamp_col = ["timestamp_14"]
    label_col = ["label"]

    # Cast dtypes
    # df[numerical_cols] = df[numerical_cols].astype(float)
    # df[label_col] = df[label_col].astype(int)

    # Convert to numpy arrays
    features = df[feature_cols].to_numpy()
    labels = df[label_col].to_numpy().flatten()
    timestamps = df[timestamp_col].to_numpy().flatten()

    return features, labels, timestamps

In [16]:
def save_npz_for_year(features, labels, timestamps, year, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"data_{year}.npz")
    np.savez(output_path, features=features, labels=labels, timestamps=timestamps)
    print(f"✅ Saved {year} data to {output_path}")

In [17]:
# Add this before the year processing loop
def create_global_encoder(anoshift_db_path, years):
    all_categorical_data = []
    for year in years:
        df = load_data(anoshift_db_path, year)
        df = rename_columns(df)
        categorical_data = df.loc[:,['cat_' in i for i in df.columns]]
        all_categorical_data.append(categorical_data)
    
    combined_categorical = pd.concat(all_categorical_data)
    encoder = OneHotEncoder(handle_unknown='ignore')
    encoder.fit(combined_categorical)
    return encoder

In [ ]:
# Create global encoder
global_encoder = create_global_encoder(anoshift_db_path, dataset_years)

for year in [2014, 2015]:

    print(f"Loading {year} dataset...")
    # Load the dataset
    # dfs = []
    df_year = load_data(anoshift_db_path, year)
    # dfs.append(df_year)
    print(f"Loaded {year} dataset with shape: {df_year.shape}")

    # print("Concatenating all years...")
    # df_all_years = pd.concat(dfs, ignore_index=True)
    # print("Concatenated DataFrame shape:", df_all_years.shape)

    # Rename columns
    print("Renaming columns...")
    df_renamed = rename_columns(df_year)
    print("Renamed DataFrame:", df_renamed)

    print("Preprocessing data...")
    df_preprocessed, _ = preprocess(df_renamed, enc=global_encoder)
    print("Preprocessed DataFrame:", df_preprocessed)

    print("Extracting features, labels, and timestamps...")
    features, labels, timestamps = extract_features_labels_timestamps(df_preprocessed)
    print("Feature shape:", features.shape)
    print("Label shape:", labels.shape)
    print("Timestamp shape:", timestamps.shape)


    print("Saving data to NPZ files...")
    # save_npz_for_year(features, labels, timestamps, year, output_dir="processed_data_npz")
    save_npz_for_year(features, labels, timestamps, year, output_dir="dataset_npz_files")
    print("Data saved successfully.")

Renamed columns:        cat_0  cat_1 cat_2 cat_3  num_4  num_5  num_6  num_7  num_8  num_9  \
0       c015  other   c20   c30      4    1.0   1.00   0.80     90    100   
1        c00  other   c20   c30      0    0.0   0.00   0.00      6     80   
2       c021   smtp  c274  c357      2    1.0   0.00   0.00      7     97   
3        c07   smtp  c274  c357     22    1.0   0.00   0.00     17     95   
4       c068   smtp  c220  c342      0    0.0   0.00   0.00      1     35   
...      ...    ...   ...   ...    ...    ...    ...    ...    ...    ...   
466769  c014  other   c20   c30      5    1.0   0.80   0.50      0    100   
466770   c04  other  c238  c338     34    1.0   0.32   0.33      0     58   
466771   c03  other  c245  c341      3    1.0   0.00   0.00     42     42   
466772  c014  other   c20   c30     77    1.0   0.88   0.71      0    100   
466773   c03  other   c20   c30      0    0.0   0.00   1.00     50     50   

        num_10  num_11  num_12 cat_13       timestamp_14  

/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_15910/2953266764.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_new['label'].replace({1:0}, inplace=True)
/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_15910/2953266764.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

        cat_0  cat_1 cat_2 cat_3  num_4  num_5  num_6  num_7  num_8  num_9  \
0         c00    dns  c239  c346      1    1.0   1.00   0.33      8     56   
1         c00    dns   c20   c30      8    1.0   0.00   0.05     10     12   
2         c00    dns  c238  c348     58    1.0   0.02   0.09     13     85   
3         c00    dns   c20   c30      6    1.0   0.17   0.26     18     18   
4         c00    dns  c237  c346      8    1.0   0.00   0.00     82     82   
...       ...    ...   ...   ...    ...    ...    ...    ...    ...    ...   
2122657   c00  other   c20   c30      6    1.0   1.00   0.12      1      2   
2122658   c00    dns  c244  c350     10    1.0   0.00   0.14     79     85   
2122659   c00  other   c20   c30     10    1.0   0.10   0.29      4     40   
2122660   c00    dns   c20   c30      3    1.0   1.00   0.57     16     46   
2122661   c00    dns  c243  c350      5    1.0   0.00   0.09     48     84   

         ...  catnum_574  catnum_575  catnum_576 catnum_577 cat

/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_15910/2953266764.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_new['label'].replace({1:0}, inplace=True)
/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_15910/2953266764.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

        cat_0  cat_1 cat_2 cat_3  num_4  num_5  num_6  num_7  num_8  num_9  \
0         c02    ssh  c265  c377     11   0.18   0.82   0.00     97     97   
1         c00    dns  c239  c348     12   1.00   0.00   0.00     14     93   
2         c00    dns  c241  c356      2   1.00   0.00   0.00      1     24   
3         c00    dns  c239  c346     28   1.00   0.18   0.25     69     69   
4         c00    dns  c240  c349      4   1.00   0.00   0.00     19     79   
...       ...    ...   ...   ...    ...    ...    ...    ...    ...    ...   
2104317  c014  other   c20   c30      1   1.00   1.00   0.56     25     61   
2104318  c014  other   c20   c30      1   1.00   1.00   0.50     48     76   
2104319  c014  other   c20   c30      1   1.00   1.00   0.60     24     94   
2104320  c015  other   c20   c30      5   1.00   1.00   0.70      1     66   
2104321  c030  other   c20   c30      1   0.00   0.00   0.75      2      2   

         ...  catnum_574  catnum_575  catnum_576 catnum_577 cat

In [ ]:
def merge_npz_files(input_dir, output_dir):
    all_features = []
    all_labels = []
    all_timestamps = []

    for file in sorted(os.listdir(input_dir)):
        print(f"Processing {file}")
        if file.endswith(".npz"):
            path = os.path.join(input_dir, file)
            with np.load(path, allow_pickle=True) as data:
                print(data)
                all_features.append(data["features"])
                all_labels.append(data["labels"])
                all_timestamps.append(data["timestamps"])
            print(f"📦 Loaded {file}")

    merged_features = np.vstack(all_features)
    merged_labels = np.concatenate(all_labels)
    merged_timestamps = np.concatenate(all_timestamps)

    output_path = os.path.join(output_dir, "consolidated_dataset.npz")
    np.savez(output_path, features=merged_features, labels=merged_labels, timestamps=merged_timestamps)
    print(f"✅ Merged file saved to {output_path}")

In [ ]:
merge_npz_files("dataset_npz_files", output_dir="dataset_npz_files/merged")

Processing data_2006.npz
NpzFile 'processed_data_npz/data_2006.npz' with keys: features, labels, timestamps
📦 Loaded data_2006.npz
Processing data_2007.npz
NpzFile 'processed_data_npz/data_2007.npz' with keys: features, labels, timestamps
📦 Loaded data_2007.npz
Processing data_2008.npz
NpzFile 'processed_data_npz/data_2008.npz' with keys: features, labels, timestamps
📦 Loaded data_2008.npz
Processing data_2009.npz
NpzFile 'processed_data_npz/data_2009.npz' with keys: features, labels, timestamps
📦 Loaded data_2009.npz
Processing data_2010.npz
NpzFile 'processed_data_npz/data_2010.npz' with keys: features, labels, timestamps
📦 Loaded data_2010.npz
Processing data_2011.npz
NpzFile 'processed_data_npz/data_2011.npz' with keys: features, labels, timestamps
📦 Loaded data_2011.npz
Processing data_2012.npz
NpzFile 'processed_data_npz/data_2012.npz' with keys: features, labels, timestamps
📦 Loaded data_2012.npz
Processing data_2013.npz
NpzFile 'processed_data_npz/data_2013.npz' with keys: feat